# Esportare schede PDF e immagini casuali in due cartelle piatte

Questo notebook esegue entrambe le operazioni nello stesso ciclo:

1. cerca ricorsivamente le cartelle dati che contengono almeno un PDF;
2. prende il primo PDF alfabetico di ogni cartella;
3. legge il codice ICCD dalle prime pagine;
4. trova la corrispondenza nel CSV;
5. recupera `id_scheda_originale`;
6. converte la prima pagina del PDF in PNG e la salva in:

```text
1.output_png_schede/<id_scheda_originale>.png
```

7. sceglie casualmente un'immagine originale dalla stessa cartella e la copia in:

```text
output_images_random/<id_scheda_originale>.<estensione_originale>
```

Sono accettate immagini `.jpg`, `.jpeg`, `.png`, `.webp`, `.tif`, `.tiff`, `.bmp` e `.gif`.

Se non esiste una corrispondenza univoca nel CSV, non viene creato nessun file per quella cartella. I file originali non vengono modificati.


In [1]:
# Se PyMuPDF non è installato, eseguire una sola volta:
# %pip install pymupdf

from pathlib import Path
import json
import random
import re
import shutil

import fitz  # PyMuPDF
import pandas as pd


In [2]:
# ==========================================================
# CONFIGURAZIONE
# ==========================================================

# Cartella esterna con le cartelle regionali e i singoli punti.
ROOT_FOLDER = Path(r"0.original_foto_schede")

# CSV contenente id_scheda_originale e _iccd_code.
CSV_PATH = Path(r"../data/dati_schede.csv")

# Cartella con la prima pagina delle schede PDF convertita in PNG.
PDF_OUTPUT_FOLDER = Path(r"1.output_png_schede")

# Cartella piatta con una fotografia/immagine casuale per scheda.
RANDOM_IMAGE_OUTPUT_FOLDER = Path(r"2.output_images_random")

# Report complessivo.
REPORT_PATH = Path(r"export_report.csv")

# Manifest degli ID disponibili per l'applicazione web.
PDF_MANIFEST_PATH = PDF_OUTPUT_FOLDER / "png-manifest.json"
IMAGE_MANIFEST_PATH = RANDOM_IMAGE_OUTPUT_FOLDER / "image-manifest.json"

# Impostazioni PDF.
PAGE_TO_EXPORT = 0
PAGES_TO_READ = 1
PNG_DPI = 200

# Impostazioni selezione casuale.
RANDOM_SEED = 30

# False evita di sovrascrivere file già creati.
OVERWRITE = False

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".JPG",
    ".JPEG",
    ".PNG"
}

print("Sorgente:", ROOT_FOLDER)
print("CSV:", CSV_PATH)
print("Schede PNG:", PDF_OUTPUT_FOLDER)
print("Immagini casuali:", RANDOM_IMAGE_OUTPUT_FOLDER)


Sorgente: 0.original_foto_schede
CSV: ../data/dati_schede.csv
Schede PNG: 1.output_png_schede
Immagini casuali: 2.output_images_random


In [3]:
# ==========================================================
# CONTROLLI INIZIALI
# ==========================================================

if not ROOT_FOLDER.is_dir():
    raise NotADirectoryError(
        f"Cartella sorgente non trovata: {ROOT_FOLDER}"
    )

if not CSV_PATH.is_file():
    raise FileNotFoundError(
        f"CSV non trovato: {CSV_PATH}"
    )

PDF_OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
RANDOM_IMAGE_OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

rng = random.Random(RANDOM_SEED)

print("Percorsi validi e cartelle di output pronte.")


Percorsi validi e cartelle di output pronte.


In [5]:
df = pd.read_csv(
    CSV_PATH,
    low_memory=False
)


def normalize_column_name(value):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).lower()
    )


def find_column(columns, required_parts):
    parts = [
        normalize_column_name(part)
        for part in required_parts
    ]

    for column in columns:
        normalized = normalize_column_name(
            column
        )

        if all(
            part in normalized
            for part in parts
        ):
            return column

    return None


ID_COLUMN = find_column(
    df.columns,
    ["id_scheda_originale"]
)

REGION_COLUMN = find_column(
    df.columns,
    ["nctr", "codice", "regione"]
)

CATALOG_COLUMN = find_column(
    df.columns,
    [
        "nctn",
        "numero",
        "catalogo",
        "generale"
    ]
)

print("Colonna UUID:", ID_COLUMN)
print("Colonna regione ICCD:", REGION_COLUMN)
print("Colonna catalogo ICCD:", CATALOG_COLUMN)

if not all(
    [
        ID_COLUMN,
        REGION_COLUMN,
        CATALOG_COLUMN
    ]
):
    raise KeyError(
        "Non sono state trovate automaticamente "
        "tutte le colonne richieste nel CSV."
    )


Colonna UUID: id_scheda_originale
Colonna regione ICCD: nctcodice_univoco_iccd__nctrcodice_regione
Colonna catalogo ICCD: nctcodice_univoco_iccd__nctnnumero_catalogo_generale


In [6]:
def integer_text(value):
    """
    Converte valori come:
    13, 13.0, '13.0', '00311644'
    in stringhe intere normalizzate.
    """
    if pd.isna(value):
        return None

    text = str(value).strip()

    if not text:
        return None

    try:
        return str(
            int(float(text))
        )

    except (
        TypeError,
        ValueError,
        OverflowError
    ):
        digits = re.sub(
            r"\D",
            "",
            text
        )

        return digits or None


regions = df[
    REGION_COLUMN
].map(integer_text)

catalogs = df[
    CATALOG_COLUMN
].map(integer_text)

iccd_codes = []

for region, catalog in zip(
    regions,
    catalogs
):
    if (
        isinstance(region, str)
        and isinstance(catalog, str)
    ):
        code = (
            region.zfill(2) +
            catalog.zfill(8)
        )

        iccd_codes.append(code)

    else:
        iccd_codes.append(pd.NA)

df["_iccd_code"] = iccd_codes


In [7]:
df.head()

,id_scheda_originale,cdidentificazione__tsktipo_scheda,cdidentificazione__lirlivello_catalogazione,nctcodice_univoco_iccd__nctrcodice_regione,nctcodice_univoco_iccd__nctnnumero_catalogo_generale,nctcodice_univoco_iccd__escente_schedatore,nctcodice_univoco_iccd__ecpente_competente_per_tutela,ogdefinizione_denominazione__ambambito_di_tutela_mic,ogdefinizione_denominazione__ctgcategoria,ogdefinizione_denominazione__ogtdefinizione_tipologica,...,accaltro_codice__accwindirizzo_web,prepreesistenze__preedenominazione,prepreesistenze__nscnotizie_storico_critiche,prepreesistenze__naiconsiderazioni_sugli_aspetti_di_interesse,prepreesistenze__prennote,csdati_catastali__cteelementi_confinanti,gpbbase_cartografica__gpbuindirizzo_web_(url),csdati_catastali__ctnnote,gegeoreferenziazione__gennote,_iccd_code
0,ce9c9265-54dd-4779-9c4b-cc282dc908e3,AR,P,6.0,182407.0,ICCD,S239,architettonico e paesaggistico,EDIFICIO SINGOLO,casa con loggiato,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0600182407
1,6e02ff7c-2b9d-40d0-a14c-e8f4d03d9a83,AR,I,20.0,252733.0,ICCD,S252,architettonico e paesaggistico,EDIFICIO SINGOLO,casa a corte chiusa,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2000252733
2,814d64dd-d0f5-4c01-ace8-968ec355547e,AR,I,17.0,223849.0,ICCD,S284,architettonico e paesaggistico,EDIFICIO CON ANNESSI,casa a scala esterna,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1700223849
3,7aea73d3-286c-4fa1-aade-0883bd9f9c78,AR,P,6.0,182399.0,ICCD,S239,architettonico e paesaggistico,EDIFICIO CON ANNESSI,cascina,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0600182399
4,d8d256d7-e480-40c6-bd8a-0d8d4dd66f45,AR,I,13.0,310548.0,ICCD,S531,architettonico e paesaggistico,COMPLESSO,casa a edifici affiancati,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1300310548


In [8]:

missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise KeyError(
        "Colonne mancanti nel CSV: "
        + ", ".join(sorted(missing_columns))
    )

def normalize_iccd_code(value):
    if pd.isna(value):
        return pd.NA

    digits = re.sub(r"\D", "", str(value))

    if not digits:
        return pd.NA

    # Protegge anche dai valori letti come 1300311644.0.
    if len(digits) > 10 and digits.endswith("0"):
        digits = digits[:-1]

    return digits.zfill(10)

def normalize_id(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).strip()

    if not text or text.lower() in {"nan", "none", "<na>"}:
        return pd.NA

    return text

df["_iccd_code_clean"] = df["_iccd_code"].map(
    normalize_iccd_code
)

df["_id_clean"] = df["id_scheda_originale"].map(
    normalize_id
)

display(
    df[
        [
            "_iccd_code",
            "_iccd_code_clean",
            "id_scheda_originale",
            "_id_clean",
        ]
    ].head()
)


,_iccd_code,_iccd_code_clean,id_scheda_originale,_id_clean
0,0600182407,0600182407,ce9c9265-54dd-4779-9c4b-cc282dc908e3,ce9c9265-54dd-4779-9c4b-cc282dc908e3
1,2000252733,2000252733,6e02ff7c-2b9d-40d0-a14c-e8f4d03d9a83,6e02ff7c-2b9d-40d0-a14c-e8f4d03d9a83
2,1700223849,1700223849,814d64dd-d0f5-4c01-ace8-968ec355547e,814d64dd-d0f5-4c01-ace8-968ec355547e
3,0600182399,0600182399,7aea73d3-286c-4fa1-aade-0883bd9f9c78,7aea73d3-286c-4fa1-aade-0883bd9f9c78
4,1300310548,1300310548,d8d256d7-e480-40c6-bd8a-0d8d4dd66f45,d8d256d7-e480-40c6-bd8a-0d8d4dd66f45


In [9]:
# ==========================================================
# LOOKUP: CODICE ICCD -> ID CSV
# ==========================================================

lookup = {}

valid_rows = df.loc[
    df["_iccd_code_clean"].notna()
    & df["_id_clean"].notna()
]

for csv_index, row in valid_rows.iterrows():
    lookup.setdefault(
        row["_iccd_code_clean"],
        [],
    ).append({
        "csv_index": csv_index,
        "id": row["_id_clean"],
    })

duplicate_codes = {
    code: matches
    for code, matches in lookup.items()
    if len(matches) > 1
}

print("Codici ICCD nel lookup:", f"{len(lookup):,}")
print("Codici ICCD duplicati:", f"{len(duplicate_codes):,}")


Codici ICCD nel lookup: 43,263
Codici ICCD duplicati: 0


In [10]:
# ==========================================================
# ESTRAZIONE DEL CODICE ICCD DAL PDF
# ==========================================================

TEN_DIGIT_CODE = re.compile(r"(?<!\d)(\d{10})(?!\d)")

def extract_iccd_code(pdf_path, pages_to_read=2):
    document = fitz.open(pdf_path)

    try:
        text = "\n".join(
            document.load_page(page_number).get_text("text")
            for page_number in range(
                min(pages_to_read, document.page_count)
            )
        )

        codes = TEN_DIGIT_CODE.findall(text)

        if not codes:
            return None

        unique_codes = list(dict.fromkeys(codes))

        # Usa soltanto codici realmente presenti nel CSV.
        valid_codes = [
            code
            for code in unique_codes
            if code in lookup
        ]

        if len(valid_codes) == 1:
            return valid_codes[0]

        if len(valid_codes) > 1:
            frequencies = pd.Series(codes).value_counts()

            return max(
                valid_codes,
                key=lambda code: frequencies.get(code, 0),
            )

        return None

    finally:
        document.close()


In [11]:
# ==========================================================
# FUNZIONI DI ESPORTAZIONE
# ==========================================================

def export_pdf_page_to_png(
    pdf_path,
    output_path,
    page_number=0,
    dpi=200,
):
    document = fitz.open(pdf_path)

    try:
        if document.page_count == 0:
            raise ValueError("Il PDF non contiene pagine.")

        if page_number >= document.page_count:
            raise IndexError(
                f"Pagina {page_number + 1} non presente nel PDF."
            )

        page = document.load_page(page_number)
        zoom = dpi / 72
        pixmap = page.get_pixmap(
            matrix=fitz.Matrix(zoom, zoom),
            alpha=False,
        )
        pixmap.save(output_path)

    finally:
        document.close()


def image_files_in_folder(folder):
    # Ricerca ricorsiva dentro la singola cartella dati.
    return sorted(
        (
            path
            for path in folder.rglob("*")
            if path.is_file()
            and path.suffix.lower() in IMAGE_EXTENSIONS
        ),
        key=lambda path: str(path).lower(),
    )


In [12]:
# ==========================================================
# INDIVIDUAZIONE DELLE CARTELLE DATI
# ==========================================================

all_pdfs = sorted(
    ROOT_FOLDER.rglob("*.pdf"),
    key=lambda path: str(path).lower(),
)

pdfs_by_folder = {}

for pdf_path in all_pdfs:
    pdfs_by_folder.setdefault(
        pdf_path.parent,
        [],
    ).append(pdf_path)

first_pdfs = sorted(
    (
        sorted(
            pdf_paths,
            key=lambda path: path.name.lower(),
        )[0]
        for pdf_paths in pdfs_by_folder.values()
    ),
    key=lambda path: str(path).lower(),
)

print("PDF complessivi trovati:", f"{len(all_pdfs):,}")
print("Cartelle dati con PDF:", f"{len(first_pdfs):,}")


PDF complessivi trovati: 593
Cartelle dati con PDF: 482


In [13]:
# ==========================================================
# ASSOCIAZIONE ED ESPORTAZIONE DEI DUE FILE
# ==========================================================

results = []
used_pdf_names = set()
used_image_ids = set()

for position, pdf_path in enumerate(first_pdfs, start=1):
    data_folder = pdf_path.parent

    record = {
        "folder": str(data_folder.relative_to(ROOT_FOLDER)),
        "pdf_name": pdf_path.name,
        "pdf_path": str(pdf_path),
        "iccd_code": pd.NA,
        "csv_index": pd.NA,
        "id_scheda_originale": pd.NA,
        "pdf_png_name": pd.NA,
        "pdf_png_path": pd.NA,
        "pdf_status": pd.NA,
        "image_candidates": 0,
        "selected_image_name": pd.NA,
        "selected_image_path": pd.NA,
        "random_image_name": pd.NA,
        "random_image_path": pd.NA,
        "image_status": pd.NA,
        "status": pd.NA,
    }

    try:
        iccd_code = extract_iccd_code(
            pdf_path,
            PAGES_TO_READ,
        )

        if not iccd_code:
            record["status"] = "no_matching_iccd_code"
            results.append(record)
            continue

        record["iccd_code"] = iccd_code
        matches = lookup.get(iccd_code, [])

        # Nessuna o più di una corrispondenza:
        # non crea nessuno dei due file.
        if len(matches) == 0:
            record["status"] = "not_found_in_csv"
            results.append(record)
            continue

        if len(matches) > 1:
            record["status"] = "multiple_csv_matches"
            results.append(record)
            continue

        match = matches[0]
        item_id = match["id"]

        record["csv_index"] = match["csv_index"]
        record["id_scheda_originale"] = item_id

        # --------------------------------------------------
        # 1. Prima pagina PDF -> PNG
        # --------------------------------------------------

        pdf_png_name = f"{item_id}.png"
        pdf_png_path = PDF_OUTPUT_FOLDER / pdf_png_name

        record["pdf_png_name"] = pdf_png_name
        record["pdf_png_path"] = str(pdf_png_path)

        if pdf_png_name.lower() in used_pdf_names:
            record["pdf_status"] = "duplicate_output_id"

        elif pdf_png_path.exists() and not OVERWRITE:
            used_pdf_names.add(pdf_png_name.lower())
            record["pdf_status"] = "already_exists"

        else:
            export_pdf_page_to_png(
                pdf_path=pdf_path,
                output_path=pdf_png_path,
                page_number=PAGE_TO_EXPORT,
                dpi=PNG_DPI,
            )

            used_pdf_names.add(pdf_png_name.lower())
            record["pdf_status"] = "exported"

        # --------------------------------------------------
        # 2. Immagine originale casuale -> cartella piatta
        # --------------------------------------------------

        candidates = image_files_in_folder(data_folder)
        record["image_candidates"] = len(candidates)

        if not candidates:
            record["image_status"] = "no_image"

        elif item_id.lower() in used_image_ids:
            record["image_status"] = "duplicate_output_id"

        else:
            selected_image = rng.choice(candidates)
            output_extension = selected_image.suffix.lower()

            # Uniforma .jpeg in .jpg, senza ricomprimere il file.
            if output_extension == ".jpeg":
                output_extension = ".jpg"

            random_image_name = (
                f"{item_id}{output_extension}"
            )

            random_image_path = (
                RANDOM_IMAGE_OUTPUT_FOLDER
                / random_image_name
            )

            record["selected_image_name"] = selected_image.name
            record["selected_image_path"] = str(selected_image)
            record["random_image_name"] = random_image_name
            record["random_image_path"] = str(random_image_path)

            if random_image_path.exists() and not OVERWRITE:
                used_image_ids.add(item_id.lower())
                record["image_status"] = "already_exists"

            else:
                shutil.copy2(
                    selected_image,
                    random_image_path,
                )

                used_image_ids.add(item_id.lower())
                record["image_status"] = "copied"

        record["status"] = "matched"

    except Exception as error:
        record["status"] = (
            f"error: {type(error).__name__}: {error}"
        )

    results.append(record)

    if position % 50 == 0 or position == len(first_pdfs):
        print(
            f"Processate {position:,} / {len(first_pdfs):,} cartelle"
        )

results_df = pd.DataFrame(results)
display(results_df.head())


Processate 50 / 482 cartelle
Processate 100 / 482 cartelle
Processate 150 / 482 cartelle
Processate 200 / 482 cartelle
Processate 250 / 482 cartelle
Processate 300 / 482 cartelle
Processate 350 / 482 cartelle
Processate 400 / 482 cartelle
Processate 482 / 482 cartelle


,folder,pdf_name,pdf_path,iccd_code,csv_index,id_scheda_originale,pdf_png_name,pdf_png_path,pdf_status,image_candidates,selected_image_name,selected_image_path,random_image_name,random_image_path,image_status,status
0,abruzzo/1300309704_CH_Ortona_EDSING_scala est_...,1300309704_CH_Ortona_EDSING_scala est_Masseria...,0.original_foto_schede/abruzzo/1300309704_CH_O...,1300309704,6994,6d183a01-cb70-45c7-8a14-530f8670b5ce,6d183a01-cb70-45c7-8a14-530f8670b5ce.png,1.output_png_schede/6d183a01-cb70-45c7-8a14-53...,already_exists,6,CH_Ortona_MasseriaBerardi_facciataSudOvest.jpg,0.original_foto_schede/abruzzo/1300309704_CH_O...,6d183a01-cb70-45c7-8a14-530f8670b5ce.jpg,2.output_images_random/6d183a01-cb70-45c7-8a14...,copied,matched
1,abruzzo/1300309779_PE_Mosciano Sant_Angelo_EDN...,"Mosciano_Masseria Tattoni, _EDANN_casa scala e...",0.original_foto_schede/abruzzo/1300309779_PE_M...,NaN,<NA>,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,no_matching_iccd_code
2,abruzzo/1300310044_AQ_Balsorano_EDSING_casa pe...,1300310044_AQ_Balsorano_EDSING_casa pendio_Alf...,0.original_foto_schede/abruzzo/1300310044_AQ_B...,1300310044,7320,73c1a68c-2463-4368-8601-caf96d07678f,73c1a68c-2463-4368-8601-caf96d07678f.png,1.output_png_schede/73c1a68c-2463-4368-8601-ca...,already_exists,4,AQ_Balsorano_Casa_Alfonsi_parete_sudest.jpg,0.original_foto_schede/abruzzo/1300310044_AQ_B...,73c1a68c-2463-4368-8601-caf96d07678f.jpg,2.output_images_random/73c1a68c-2463-4368-8601...,copied,matched
3,abruzzo/1300310088_CH_Casoli_EDSING_scala est,1300310088_CH_Casoli_EDSING_scala est.pdf,0.original_foto_schede/abruzzo/1300310088_CH_C...,1300310088,7344,6ad295a2-8456-478d-8919-5f3f134d8a77,6ad295a2-8456-478d-8919-5f3f134d8a77.png,1.output_png_schede/6ad295a2-8456-478d-8919-5f...,already_exists,7,CH_Casoli_Contrada_Guerenna_Nuova (3)_prospett...,0.original_foto_schede/abruzzo/1300310088_CH_C...,6ad295a2-8456-478d-8919-5f3f134d8a77.jpg,2.output_images_random/6ad295a2-8456-478d-8919...,copied,matched
4,abruzzo/1300310328_PE_Penne_EDSING_casa pendio...,1300310328_PE_Penne_EDSING_casa pendio_Formica...,0.original_foto_schede/abruzzo/1300310328_PE_P...,1300310328,24581,bbd50aa9-fd4e-4ba6-8e18-05db10aa7726,bbd50aa9-fd4e-4ba6-8e18-05db10aa7726.png,1.output_png_schede/bbd50aa9-fd4e-4ba6-8e18-05...,already_exists,6,1_PE_Penne_Contrada_Colle_Formica (2)_prospett...,0.original_foto_schede/abruzzo/1300310328_PE_P...,bbd50aa9-fd4e-4ba6-8e18-05db10aa7726.jpg,2.output_images_random/bbd50aa9-fd4e-4ba6-8e18...,copied,matched


In [14]:
# ==========================================================
# REPORT FINALE
# ==========================================================

results_df.to_csv(
    REPORT_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Riepilogo generale:")
display(
    results_df["status"]
    .value_counts(dropna=False)
    .rename("count")
    .to_frame()
)

print("Schede PDF:")
display(
    results_df["pdf_status"]
    .value_counts(dropna=False)
    .rename("count")
    .to_frame()
)

print("Immagini casuali:")
display(
    results_df["image_status"]
    .value_counts(dropna=False)
    .rename("count")
    .to_frame()
)

print(
    "Schede PNG create:",
    f"{results_df['pdf_status'].eq('exported').sum():,}",
)
print(
    "Immagini casuali copiate:",
    f"{results_df['image_status'].eq('copied').sum():,}",
)
print("Report:", REPORT_PATH)


Riepilogo generale:


,count
status,
matched,439
no_matching_iccd_code,42
error: FileDataError: '0.original_foto_schede/piemonte/0100461070_AL_Frugarolo_La Torre_cascina corte chiusa.pdf' is no file,1


Schede PDF:


,count
pdf_status,
already_exists,438
NaN,43
duplicate_output_id,1


Immagini casuali:


,count
image_status,
copied,432
NaN,43
no_image,6
duplicate_output_id,1


Schede PNG create: 0
Immagini casuali copiate: 432
Report: export_report.csv


In [15]:
# ==========================================================
# MANIFEST PER L'APPLICAZIONE WEB
# ==========================================================

pdf_ids = sorted(
    path.stem
    for path in PDF_OUTPUT_FOLDER.glob("*.png")
)

image_files = sorted(
    (
        path
        for path in RANDOM_IMAGE_OUTPUT_FOLDER.iterdir()
        if path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    ),
    key=lambda path: path.name.lower(),
)

image_manifest = {
    path.stem: path.name
    for path in image_files
}

PDF_MANIFEST_PATH.write_text(
    json.dumps(
        pdf_ids,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

IMAGE_MANIFEST_PATH.write_text(
    json.dumps(
        image_manifest,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("Manifest schede:", PDF_MANIFEST_PATH)
print("Manifest immagini:", IMAGE_MANIFEST_PATH)
print("ID schede PNG:", len(pdf_ids))
print("Immagini casuali:", len(image_manifest))


Manifest schede: 1.output_png_schede/png-manifest.json
Manifest immagini: 2.output_images_random/image-manifest.json
ID schede PNG: 438
Immagini casuali: 432


In [16]:
# Mostra soltanto i casi incompleti o problematici.

problems_df = results_df.loc[
    (results_df["status"] != "matched")
    | ~results_df["pdf_status"].isin(
        ["exported", "already_exists"]
    )
    | ~results_df["image_status"].isin(
        ["copied", "already_exists"]
    )
].copy()

display(problems_df)


,folder,pdf_name,pdf_path,iccd_code,csv_index,id_scheda_originale,pdf_png_name,pdf_png_path,pdf_status,image_candidates,selected_image_name,selected_image_path,random_image_name,random_image_path,image_status,status
1,abruzzo/1300309779_PE_Mosciano Sant_Angelo_EDN...,"Mosciano_Masseria Tattoni, _EDANN_casa scala e...",0.original_foto_schede/abruzzo/1300309779_PE_M...,NaN,<NA>,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,no_matching_iccd_code
20,basilicata/1700223900_MT_Matera_complesso_mass...,download (13).pdf,0.original_foto_schede/basilicata/1700223900_M...,NaN,<NA>,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,no_matching_iccd_code
62,campania/1500957067_Grazzanise_CE_edificio con...,download (13).pdf,0.original_foto_schede/campania/1500957067_Gra...,NaN,<NA>,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,no_matching_iccd_code
96,lazio/Arce (FR)_ Casale Pantanaccio via Campos...,download.pdf,0.original_foto_schede/lazio/Arce (FR)_ Casale...,NaN,<NA>,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,no_matching_iccd_code
113,lazio/Castel Sant_Angelo (RI)_ Via Salaria_EDI...,download.pdf,0.original_foto_schede/lazio/Castel Sant_Angel...,NaN,<NA>,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,no_matching_iccd_code
121,lazio/Civitella d_Agliano (VT)_ Strada Provinc...,download.pdf,0.original_foto_schede/lazio/Civitella d_Aglia...,1201417725,37878,3f14ea78-6757-45df-ab4e-f06883d0f495,3f14ea78-6757-45df-ab4e-f06883d0f495.png,1.output_png_schede/3f14ea78-6757-45df-ab4e-f0...,already_exists,0,NaN,NaN,NaN,NaN,no_image,matched
122,"lazio/Colfelice (FR)_ via Casilina 12_ED.ANN.,...",download.pdf,0.original_foto_schede/lazio/Colfelice (FR)_ v...,NaN,<NA>,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,no_matching_iccd_code
126,lazio/Fiumicino (RM)_ Centro Falconieri Via de...,Centro Falconieri_ortofoto_1.pdf,0.original_foto_schede/lazio/Fiumicino (RM)_ C...,NaN,<NA>,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,no_matching_iccd_code
129,lazio/Fiumicino (RM)_ Centro Granaretto_Bonifi...,Centro_Granaretto_ortofoto.pdf,0.original_foto_schede/lazio/Fiumicino (RM)_ C...,NaN,<NA>,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,no_matching_iccd_code
135,lazio/Fontana Liri (FR)_ Via Cafenna 2_EDIFICI...,download.pdf,0.original_foto_schede/lazio/Fontana Liri (FR)...,NaN,<NA>,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,no_matching_iccd_code
